In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
VOCAB_SIZE = 12          # tokens 0-9 = data, 10 = <START>, 11 = <END>
START_TOKEN = 10
END_TOKEN = 11
SEQ_LEN = 8              # length of the digit sequence
NUM_TRAIN = 5000

rng = np.random.default_rng(0)

src = rng.integers(0, 10, size=(NUM_TRAIN, SEQ_LEN))
tgt = src[:, ::-1]   # reversed version of each row

print("source:", src[0])
print("target:", tgt[0])

source: [8 6 5 2 3 0 0 0]
target: [0 0 0 3 2 5 6 8]


In [10]:


dec_in = np.concatenate([np.full((NUM_TRAIN, 1), START_TOKEN), tgt], axis=1)
dec_out = np.concatenate([tgt, np.full((NUM_TRAIN, 1), END_TOKEN)], axis=1)

print("target     :", tgt[0])
print("decoder in :", dec_in[0])
print("decoder out:", dec_out[0])

target     : [0 0 0 3 2 5 6 8]
decoder in : [10  0  0  0  3  2  5  6  8]
decoder out: [ 0  0  0  3  2  5  6  8 11]


In [11]:
class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model):
        super().__init__()
        pos = np.arange(max_len)[:, np.newaxis]
        i = np.arange(d_model)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
        angle_rads = pos * angle_rates
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        self.pos_encoding = tf.cast(angle_rads[np.newaxis, ...], dtype=tf.float32)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pos_encoding[:, :seq_len, :]

In [12]:
class EncoderBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn = keras.Sequential([layers.Dense(ff_dim, activation="relu"), layers.Dense(d_model)])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn_out = self.mha(query=x, key=x, value=x)
        x = self.norm1(x + self.drop1(attn_out, training=training))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.drop2(ffn_out, training=training))
        return x

In [13]:
class DecoderBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.self_mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.cross_mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn = keras.Sequential([layers.Dense(ff_dim, activation="relu"), layers.Dense(d_model)])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.norm3 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)
        self.drop3 = layers.Dropout(dropout)

    def call(self, x, enc_output, training=False):
        self_attn_out = self.self_mha(query=x, key=x, value=x, use_causal_mask=True)
        x = self.norm1(x + self.drop1(self_attn_out, training=training))

        cross_attn_out = self.cross_mha(query=x, key=enc_output, value=enc_output)
        x = self.norm2(x + self.drop2(cross_attn_out, training=training))

        ffn_out = self.ffn(x)
        x = self.norm3(x + self.drop3(ffn_out, training=training))
        return x

In [14]:
D_MODEL, NUM_HEADS, FF_DIM, NUM_LAYERS, DROPOUT = 64, 4, 128, 2, 0.1
MAX_LEN = SEQ_LEN + 1

class EncoderDecoderTransformer(keras.Model):
    def __init__(self):
        super().__init__()
        self.enc_embed = layers.Embedding(VOCAB_SIZE, D_MODEL)
        self.dec_embed = layers.Embedding(VOCAB_SIZE, D_MODEL)
        self.pos_enc = PositionalEncoding(MAX_LEN, D_MODEL)
        self.enc_blocks = [EncoderBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT) for _ in range(NUM_LAYERS)]
        self.dec_blocks = [DecoderBlock(D_MODEL, NUM_HEADS, FF_DIM, DROPOUT) for _ in range(NUM_LAYERS)]
        self.final_linear = layers.Dense(VOCAB_SIZE)

    def call(self, inputs, training=False):
        src, dec_in = inputs
        x_enc = self.pos_enc(self.enc_embed(src))
        for block in self.enc_blocks:
            x_enc = block(x_enc, training=training)

        x_dec = self.pos_enc(self.dec_embed(dec_in))
        for block in self.dec_blocks:
            x_dec = block(x_dec, x_enc, training=training)

        return self.final_linear(x_dec)

model = EncoderDecoderTransformer()
model.compile(optimizer="adam",
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])
_ = model((src[:2], dec_in[:2]))
model.summary()

Model: "encoder_decoder_transformer_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ (2, 8, 64)                  │             768 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding_3 (Embedding)              │ (2, 9, 64)                  │             768 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ positional_encoding_1                │ ?                           │               0 │
│ (PositionalEncoding)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ encoder_block_2 (EncoderBlock)       │ ?                           │          33,472 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ encoder_block_3 (EncoderBlock)       │ ?                           │          33,472 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ decoder_block_2 (DecoderBlock)       │ ?                           │          50,240 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ decoder_block_3 (DecoderBlock)       │ ?                           │          50,240 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_17 (Dense)                     │ (2, 9, 12)                  │             780 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 169,740 (663.05 KB)

 Trainable params: 169,740 (663.05 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit((src, dec_in), dec_out, validation_split=0.1, epochs=15, batch_size=64)

Epoch 1/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 51s 117ms/step - accuracy: 0.1977 - loss: 2.1837 - val_accuracy: 0.3187 - val_loss: 1.8911
Epoch 2/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 6s 77ms/step - accuracy: 0.7610 - loss: 0.7020 - val_accuracy: 1.0000 - val_loss: 0.0025
Epoch 3/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 5s 69ms/step - accuracy: 0.9999 - loss: 0.0046 - val_accuracy: 1.0000 - val_loss: 9.7786e-04
Epoch 4/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 5s 74ms/step - accuracy: 1.0000 - loss: 0.0027 - val_accuracy: 1.0000 - val_loss: 6.3480e-04
Epoch 5/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 5s 72ms/step - accuracy: 0.9998 - loss: 0.0021 - val_accuracy: 1.0000 - val_loss: 6.0225e-04
Epoch 6/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 5s 74ms/step - accuracy: 0.9777 - loss: 0.0966 - val_accuracy: 1.0000 - val_loss: 8.3349e-04
Epoch 7/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 5s 75ms/step - accuracy: 0.9998 - loss: 0.0021 - val_accuracy: 1.0000 - val_loss: 5.0453e-04
Epoch 8/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 5s 74ms/step - accuracy: 1.0000 - loss: 0.0013 - va

In [ ]:
def greedy_decode(src_seq):
    src_seq = np.array(src_seq).reshape(1, -1)
    dec_input = [START_TOKEN]
    for _ in range(SEQ_LEN + 1):
        dec_in_arr = np.array(dec_input).reshape(1, -1)
        logits = model((src_seq, dec_in_arr), training=False)
        next_token = int(tf.argmax(logits[0, -1]).numpy())
        if next_token == END_TOKEN:
            break
        dec_input.append(next_token)
    return dec_input[1:]

for i in range(5):
    predicted = greedy_decode(src[i])
    print("input:", list(src[i]), "| predicted:", predicted, "| expected:", list(src[i][::-1]))

input: [np.int64(8), np.int64(6), np.int64(5), np.int64(2), np.int64(3), np.int64(0), np.int64(0), np.int64(0)] | predicted: [0, 0, 0, 3, 2, 5, 6, 8] | expected: [np.int64(0), np.int64(0), np.int64(0), np.int64(3), np.int64(2), np.int64(5), np.int64(6), np.int64(8)]
input: [np.int64(1), np.int64(8), np.int64(6), np.int64(9), np.int64(5), np.int64(6), np.int64(9), np.int64(7)] | predicted: [7, 9, 6, 5, 9, 6, 8, 1] | expected: [np.int64(7), np.int64(9), np.int64(6), np.int64(5), np.int64(9), np.int64(6), np.int64(8), np.int64(1)]
